# Tutorial – Session 10

## Goodness-of-fit og analyse af kategoriske data (Ross kap. 11.1–11.4)

Denne tutorial følger **Ross, kapitel 11**, afsnit **11.1** (introduktion), **11.2** (goodness-of-fit når alle parametre i $H_0$ er specificeret), **11.3** (goodness-of-fit når nogle parametre er uspecificeret og må estimeres fra data), og **11.4** (test for uafhængighed i kontingenstabeller).

Vi introducerer også en 5. testtype til rå kontinuerte data ved hjælp af `scipy.stats.goodness_of_fit`.

**Notebook-version:** [Tutorial_10_notebook.ipynb](Tutorial_10_notebook.ipynb)

---

## 1. Goodness-of-fit for en fast fordeling (fuldt specificeret)

Vi tester om observerede tællinger passer til en specifik fordeling, hvor alle parametre er kendte. Vi bruger Pearsons $\chi^2$-teststørrelse:

$$ \chi^2 = \sum \frac{(O_i - E_i)^2}{E_i} $$

Hvor $O_i$ er de observerede frekvenser og $E_i$ er de forventede frekvenser under $H_0$.
Frihedsgrader: $k - 1$, hvor $k$ er antallet af kategorier.

Vi bruger `scipy.stats.chisquare`.

In [1]:
import numpy as np
from scipy import stats

# Eksempel: Markedsandele for 4 forskellige brands (A, B, C, D)
# Forventede markedsandele under H0 (f.eks. fra sidste år)
p_expected = np.array([0.50, 0.25, 0.15, 0.10])

# Observerede tællinger i en ny stikprøve (n = 400)
observed = np.array([210, 90, 65, 35])
n = observed.sum()

# Forventede tællinger
expected = n * p_expected

# Udfør Chi-i-anden test
chi2_stat, p_value = stats.chisquare(f_obs=observed, f_exp=expected)

print(f"Observerede: {observed}")
print(f"Forventede : {expected}")
print(f"Chi2-statistik: {chi2_stat:.4f}")
print(f"p-værdi       : {p_value:.4f}")

Observerede: [210  90  65  35]
Forventede : [200. 100.  60.  40.]
Chi2-statistik: 2.5417
p-værdi       : 0.4678


---

## 2. Goodness-of-fit for en fordeling med kendt parameter

Her tester vi om data følger en bestemt fordeling, f.eks. en Poisson-fordeling, hvor parameteren er givet på forhånd (ikke estimeret fra data). Frihedsgraderne er stadig $k - 1$.

In [10]:
# Eksempel: Kunder der ankommer til en bank pr. time.
# Vi forventer en Poisson-fordeling med lambda = 3.5
# Antal kunder: 0, 1, 2, 3, 4, 5, 6, 7 eller flere
# Observerede timer (n = 100)
obs_hours = np.array([3, 12, 20, 25, 18, 12, 7, 3])
n_hours = obs_hours.sum()
k = len(obs_hours)

# Forventede sandsynligheder under Poisson(3.5)
lam = 3.5
p_poisson = np.array([stats.poisson.pmf(i, lam) for i in range(k)])

# Da den sidste kategori er "7 eller flere", samler vi resten af sandsynligheden her
p_poisson[-1] = 1 - p_poisson[:-1].sum()

exp_hours = n_hours * p_poisson

print(f"Observerede: {obs_hours}")
print(f"Forventede : {np.round(exp_hours, 2)}")

Observerede: [ 3 12 20 25 18 12  7  3]
Forventede : [ 3.02 10.57 18.5  21.58 18.88 13.22  7.71  6.53]


In [12]:
# Vi ser at den første kategori er mindre end 5 og slår den sammen:

obs_pooled = np.append([obs_hours[0] + obs_hours[1]], obs_hours[2:])
exp_pooled = np.append([exp_hours[0] + exp_hours[1]], exp_hours[2:])

# Calculate Chi2 with the pooled data
chi2_stat, p_value = stats.chisquare(f_obs=obs_pooled, f_exp=exp_pooled)

print(f"Pooled Observed: {obs_pooled}")
print(f"Pooled Expected: {exp_pooled}")
print(f"Chi2-statistik: {chi2_stat:.4f}")
print(f"p-værdi       : {p_value:.4f}")

Pooled Observed: [15 20 25 18 12  7  3]
Pooled Expected: [13.58882254 18.49589735 21.5785469  18.88122854 13.21685998  7.70983499
  6.5288097 ]
Chi2-statistik: 2.9372
p-værdi       : 0.8167


---

## 3. Goodness-of-fit for en fordeling med ukendt parameter (estimeret fra data)

Hvis vi estimerer $m$ parametre fra data, mister vi yderligere $m$ frihedsgrader. De samlede frihedsgrader bliver $k - 1 - m$.
I `scipy.stats.chisquare` angives dette med parameteren `ddof` (Delta Degrees of Freedom), som sættes til $m$.

In [14]:
# Eksempel: Vægten af æbler (Normalfordeling, hvor middelværdi og standardafvigelse estimeres)
# Antag vi har n=200 observationer, inddelt i 5 intervaller.
# Vi har estimeret mu = 150g og sigma = 20g fra data.
n_obs = 200
obs_counts = np.array([15, 45, 80, 45, 15])

# Intervaller: (-inf, 120], (120, 140], (140, 160], (160, 180], (180, inf)
mu_est = 150
sigma_est = 20

# Beregn forventede sandsynligheder for hvert interval
edges = [-np.inf, 120, 140, 160, 180, np.inf]

# Beregn alle CDF værdier på én gang
cdf_values = stats.norm.cdf(edges, loc=mu_est, scale=sigma_est)

# Find forskellen mellem hvert trin (p_norm[i] = cdf[i+1] - cdf[i])
p_norm = np.diff(cdf_values)
exp_counts = n_obs * p_norm

# Vi estimerede 2 parametre (mu og sigma), så ddof = 2
chi2_stat, p_value = stats.chisquare(f_obs=obs_counts, f_exp=exp_counts, ddof=2)

print(f"Observerede: {obs_counts}")
print(f"Forventede : {np.round(exp_counts, 2)}")
print(f"Chi2-statistik: {chi2_stat:.4f}")
print(f"p-værdi (ddof=2): {p_value:.4f}")

Observerede: [15 45 80 45 15]
Forventede : [13.36 48.35 76.58 48.35 13.36]
Chi2-statistik: 1.0173
p-værdi (ddof=2): 0.6013


---

## 4. Kontingenstabel-test for uafhængighed

Når vi har to kategoriske variable, kan vi opstille en kontingenstabel (krydstabel) og teste om de to variable er uafhængige.
Vi bruger `scipy.stats.chi2_contingency`, som automatisk beregner de forventede frekvenser og frihedsgraderne $(r-1)(c-1)$.

In [15]:
# Eksempel: Sammenhæng mellem aldersgruppe og politisk præference
# Rækker: Ung, Mellem, Ældre
# Kolonner: Kandidat A, Kandidat B, Kandidat C
contingency_table = np.array([
    [45, 30, 25],
    [35, 50, 15],
    [20, 40, 40]
])

# Udfør test for uafhængighed
chi2_stat, p_value, dof, expected = stats.chi2_contingency(contingency_table, correction=False)

print(f"Chi2-statistik: {chi2_stat:.4f}")
print(f"p-værdi       : {p_value:.4f}")
print(f"Frihedsgrader : {dof}")
print("Forventede frekvenser:")
print(np.round(expected, 2))

Chi2-statistik: 26.3750
p-værdi       : 0.0000
Frihedsgrader : 4
Forventede frekvenser:
[[33.33 40.   26.67]
 [33.33 40.   26.67]
 [33.33 40.   26.67]]


---

## 5. Test for rå kontinuerte data (`stats.goodness_of_fit`)

Hvis vi har de rå kontinuerte data (ikke inddelt i intervaller/bins), kan vi bruge mere avancerede tests som f.eks. Anderson-Darling eller Kolmogorov-Smirnov.
`scipy.stats.goodness_of_fit` (introduceret i nyere versioner af SciPy) udfører en Monte Carlo-baseret goodness-of-fit test, som er meget stærk til rå data.

In [5]:
# Eksempel: Test om 50 målte højder (i cm) er normalfordelte.
np.random.seed(42)
# Vi genererer 50 tilfældige højder fra en normalfordeling for eksemplets skyld
raw_data = stats.norm.rvs(loc=175, scale=10, size=50)

# Brug stats.goodness_of_fit til at teste om data er normalfordelt.
# Funktionen estimerer selv parametrene (loc og scale) fra data.
# Vi bruger Anderson-Darling (ad) statistikken.
res = stats.goodness_of_fit(stats.norm, raw_data, statistic="ad", random_state=42)

print(f"Test-statistik (Anderson-Darling): {res.statistic:.4f}")
print(f"p-værdi: {res.pvalue:.4f}")

if res.pvalue > 0.05:
    print("Vi kan ikke afvise H0: Data ser ud til at være normalfordelt.")
else:
    print("Vi afviser H0: Data er ikke normalfordelt.")

Test-statistik (Anderson-Darling): 0.1907
p-værdi: 0.9048
Vi kan ikke afvise H0: Data ser ud til at være normalfordelt.
